
<style>
.ebdr-header,.ebdr-section{background:linear-gradient(90deg,#c00000,#ff3333);color:#fff!important;padding:18px 22px;border-radius:8px;margin:12px 0}
.ebdr-header h1,.ebdr-header h2,.ebdr-section h2{color:#fff!important;margin:0 0 8px}
.ebdr-header p,.ebdr-section p{color:#fff!important;margin:5px 0}
.ebdr-accent{color:#ffcc00!important;font-weight:700}
.ebdr-activity{background:#fff3f3;border-left:4px solid #ff3333;padding:14px 18px;margin:12px 0;border-radius:4px}
.gabarito{background:#fafafa;border:1px solid #ddd;border-radius:8px;padding:16px}
.gabarito summary{color:#c00000;font-weight:700;cursor:pointer}
.gabarito button{float:right;background:#c00000;color:white;border:0;border-radius:5px;padding:6px 10px;cursor:pointer}
#geral-code{white-space:pre-wrap;font-family:monospace;font-size:13px;max-height:650px;overflow:auto;margin-top:12px}
</style>


<div class="ebdr-header">
<h1>Modelagem de Rotor Flexível DTU - 12 Elementos</h1>
<p><span class="ebdr-accent">[INSERIR IMAGEM AQUI]</span></p>
<p><b>Análises:</b> Modal · Campbell · FRF · Unbalance · Critical Speed · Time Response</p>
<p>MACHINERY DYNAMICS LECTURES (41514) - MEK/DTU - Ilmar Ferreira Santos</p>
<p><b>Resultados experimentais:</b> 13.0, 14.9, 33.6, 43.0, 46.0 Hz</p>
</div>

<div class="ebdr-section"><h2>2. Introdução teórica</h2><p><span class="ebdr-accent">[INSERIR IMAGEM AQUI]</span></p></div>

O modelo representa um rotor flexível com **2 discos** e **2 mancais**, incluindo a dinâmica da carcaça por meio de nós adicionais conectados com `n_link`.

### Formulação

Na representação original em MATLAB são considerados **4 DOF laterais por nó**, enquanto o ROSS trabalha com **6 DOF por nó**. A conversão para o modelo lateral é realizada por `convert_6dof_to_4dof`.

O problema de vibração livre pode ser escrito como

\[
M\ddot q + (C+\Omega G)\dot q + Kq = 0,
\]

onde \(M\), \(C\), \(K\) e \(G\) são as matrizes de massa, amortecimento, rigidez e efeito giroscópico, respectivamente.

O arquivo-fonte fornecido registra o padrão `n_link` e a conversão de 6 para 4 DOF.

<div class="ebdr-section"><h2>3. Inicialização</h2></div>

In [ ]:
import ross as rs
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import linalg as la
from ross.utils import convert_6dof_to_4dof

print("Bibliotecas importadas.")

<div class="ebdr-section"><h2>4. Parâmetros</h2></div>

| Parâmetro | Valor |
|---|---:|
| NE | 12 |
| ND / NM | 2 / 2 |
| CD1 / CD2 | 3 / 9 |
| CMM1 / CMM2 | 0 / 12 |
| HOUSING1 / HOUSING2 | 13 / 14 |
| E | \(2	imes10^{11}\) Pa |
| \(ho_{steel}\) / \(ho_{Al}\) | 7800 / 2770 kg/m³ |
| \(\Omega\) | 0 rad/s |
| Rd / Ri / espD | 0.06 / 0.0025 / 0.011 m |
| MasM | 0.40698 kg |
| h / b / lr | 0.001 / 0.0285 / 0.075 m |
| Rext / Rint | 0.0025 / 0 m |

In [ ]:
NE=12; ND=2; NM=2
CD1=3; CD2=9; CMM1=0; CMM2=12
HOUSING1=13; HOUSING2=14

E=2e11; RAco=7800; RAl=2770; Omega=0
Rd=0.06; Ri=0.0025; espD=0.011
MasM=0.40698; h=0.001; b=0.0285; lr=0.075
Rext=0.0025; Rint=0.0

Area=b*h
I_beam=b*h**3/12
Kty0=2*12*E*I_beam/lr**3
Ktz0=2*E*Area/lr

print(f"Kty0 = {Kty0:.3e} N/m")
print(f"Ktz0 = {Ktz0:.3e} N/m")

<div class="ebdr-section"><h2>5. Materiais</h2></div>

Materiais usados no modelo: alumínio para os discos e aço para o eixo.

In [ ]:
aluminum=rs.Material(name="aluminum",rho=RAl,E=70e9,G_s=27e9)
steel_DTU=rs.Material(name="steel_DTU",rho=RAco,E=E,G_s=8.0e10)

<div class="ebdr-section"><h2>6. Elementos de eixo</h2></div>

\[
l=[3	imes(0.140/3),\,6	imes(0.205/6),\,3	imes(0.090/3)].
\]

São ativados `shear_effects`, `rotary_inertia` e `gyroscopic`.

In [ ]:
l=np.array([0.140/3]*3+[0.205/6]*6+[0.090/3]*3)

shaft_elements=[]
for i,Li in enumerate(l):
    shaft_elements.append(rs.ShaftElement(
        L=Li,idl=2*Rint,odl=2*Rext,material=steel_DTU,n=i,
        shear_effects=True,rotary_inertia=True,gyroscopic=True
    ))

print(f"Elementos: {len(shaft_elements)}")
print(f"Comprimento total: {np.sum(l):.4f} m")

<div class="ebdr-section"><h2>7. Discos</h2></div>

In [ ]:
disk1=rs.DiskElement.from_geometry(
    n=CD1,material=aluminum,width=espD,i_d=2*Ri,o_d=2*Rd
)
disk2=rs.DiskElement.from_geometry(
    n=CD2,material=aluminum,width=espD,i_d=2*Ri,o_d=2*Rd
)

print(f"Disco 1 no nó {CD1}")
print(f"Disco 2 no nó {CD2}")

<div class="ebdr-section"><h2>8. Mancais</h2></div>

Padrão `n_link`: mancal no nó do eixo conectado ao nó da carcaça; suporte no nó da carcaça; massa pontual na carcaça.

In [ ]:
bearing1=rs.BearingElement(
    n=CMM1,n_link=HOUSING1,kxx=Ktz0,kyy=Kty0,
    cxx=0.0,cyy=0.0,tag="Bearing1"
)
support1=rs.BearingElement(
    n=HOUSING1,kxx=Ktz0,kyy=Kty0,
    cxx=0.0,cyy=0.0,tag="Support1"
)
bearing2=rs.BearingElement(
    n=CMM2,n_link=HOUSING2,kxx=Ktz0,kyy=Kty0,
    cxx=0.0,cyy=0.0,tag="Bearing2"
)
support2=rs.BearingElement(
    n=HOUSING2,kxx=Ktz0,kyy=Kty0,
    cxx=0.0,cyy=0.0,tag="Support2"
)
point_mass1=rs.PointMass(n=HOUSING1,m=MasM,tag="HousingMass1")
point_mass2=rs.PointMass(n=HOUSING2,m=MasM,tag="HousingMass2")

<div class="ebdr-section"><h2>9. Montagem + 4 DOF</h2></div>

In [ ]:
rotor=rs.Rotor(
    shaft_elements=shaft_elements,
    disk_elements=[disk1,disk2],
    bearing_elements=[bearing1,support1,bearing2,support2],
    point_mass_elements=[point_mass1,point_mass2],
)
rotor_4dof=convert_6dof_to_4dof(rotor)

print(f"Rotor: {len(rotor.nodes)} nós, {rotor.ndof} DOF")
print(f"Rotor 4 DOF: {rotor_4dof.ndof} DOF")

M_test=rotor_4dof.M()
print(f"M shape: {M_test.shape}")
print(f"cond(M): {np.linalg.cond(M_test):.6e}")

rotor.plot_rotor(nodes=999).show()

<div class="ebdr-section"><h2>10. Análise modal</h2></div>

In [ ]:
modal=rotor_4dof.run_modal(speed=Omega)
wn_hz=modal.wn/(2*np.pi)

for i,freq in enumerate(wn_hz[:10],start=1):
    print(f"Modo {i}: {freq:.2f} Hz")

for mode_num in range(1,min(7,len(wn_hz)+1)):
    try:
        fig=modal.plot_mode_3d(mode=mode_num)
        fig.update_layout(title=f"Mode {mode_num} - f={wn_hz[mode_num-1]:.2f} Hz")
        fig.show()
    except Exception as exc:
        print(f"Modo {mode_num}: {exc}")

<div class="ebdr-section"><h2>11. Diagrama de Campbell</h2></div>

In [ ]:
speed_range=np.linspace(0,3000*2*np.pi/60,50)
campbell=rotor_4dof.run_campbell(speed_range)
campbell.plot().show()

<div class="ebdr-section"><h2>12. Velocidades críticas</h2></div>

In [ ]:
critical=rotor_4dof.run_critical_speed(num_modes=12)
print(critical)

<div class="ebdr-section"><h2>13. Resposta em frequência (FRF)</h2></div>

In [ ]:
omega=np.linspace(0.1,500,500)
freq_resp=rotor_4dof.run_freq_response(speed_range=omega)

freq_resp.plot_magnitude(inp=13,out=13).show()
freq_resp.plot_phase(inp=13,out=13).show()
freq_resp.plot_polar_bode(inp=13,out=13).show()

<div class="ebdr-section"><h2>14. Resposta a desbalanceamento</h2></div>

In [ ]:
unb_resp=rotor_4dof.run_unbalance_response(
    node=[3,9],magnitude=[0.001,0.001],phase=[0,0],frequency=omega
)
unb_resp.plot_magnitude(probe=3).show()
unb_resp.plot_bode(probe=3).show()
unb_resp.plot_deflected_shape(speed=100*2*np.pi/60).show()

<div class="ebdr-section"><h2>15. Resposta no tempo</h2></div>

In [ ]:
speed=100*2*np.pi/60
t=np.linspace(0,10,10001)
F=np.zeros((len(t),rotor_4dof.ndof),dtype=complex)

time_resp=rotor_4dof.run_time_response(speed,F,t,method="newmark")
time_resp.plot_1d(probe=3).show()
time_resp.plot_dfft(probe=3).show()

<div class="ebdr-section"><h2>16. Tabela de validação</h2></div>

| Modo | Exp (Hz) | ROSS 12-el | ROSS 24-el | Erro 12% | Erro 24% |
|---:|---:|---:|---:|---:|---:|
| 1 | 13.0 | — | — | — | — |
| 2 | 14.9 | — | — | — | — |
| 3 | 33.6 | — | — | — | — |
| 4 | 43.0 | — | — | — | — |
| 5 | 46.0 | — | — | — | — |

A fonte fornecida contém os cinco resultados experimentais, mas não fornece resultados de uma versão de 24 elementos; essa coluna deve ser obtida executando o modelo refinado.

<div class="ebdr-section"><h2>17. Atividades</h2></div>

<div class="ebdr-activity">📝 <b>Convergência 12 vs 24 elementos:</b> compare as primeiras cinco frequências naturais e os erros relativos.</div>
<div class="ebdr-activity">📝 <b>Efeito giroscópico:</b> compare Ω = 0 e Ω = 3000 RPM.</div>
<div class="ebdr-activity">📝 <b>Amortecimento nos mancais:</b> introduza cxx e cyy e avalie FRF e resposta temporal.</div>
<div class="ebdr-activity">📝 <b>Posição dos discos:</b> altere CD1/CD2 e investigue frequências e formas modais.</div>

<div class="ebdr-section"><h2>18. Gabarito</h2></div>

<details class="gabarito">
<summary><span>📋 Gabarito - Código Completo</span><button onclick="copyGabarito()">📋 Copiar</button></summary>
<div id="geral-code"></div>
</details>
<script>
document.getElementById("geral-code").innerText = "\"\"\"\nMACHINERY DYNAMICS LECTURES  (41514)\nMEK - DEPARTMENT OF MECHANICAL ENGINEERING\nDTU - TECHNICAL UNIVERSITY OF DENMARK\n\nCopenhagen, March 30th, 2021\n\nIlmar Ferreira Santos\n\nROTATING MACHINES -- NATURAL FREQUENCIES AND MODES\n\nEXPERIMENTAL RESULTS\n13.0 (horizontal)\n14.9 (vertical)\n33.6 (horizontal)\n43.0 (horizontal)\n46.0 (vertical)\n\nConverted from MATLAB to Python using ROSS (Rotordynamic Open-Source Software)\n12-element version\n\nUsing proper ROSS pattern with n_link:\n- Bearing at shaft node with n_link to housing node\n- Support bearing at housing node (stiffness to ground)\n- PointMass at housing node (housing mass)\n\"\"\"\n\nimport numpy as np\nimport pandas as pd\nimport ross as rs\nfrom scipy import linalg as la\n\n# =========================================================================\n#   DEFINITION OF THE STRUCTURE OF THE MODEL\n# =========================================================================\nNE = 12   # number of shaft elements\nND = 2    # number of discs\nNM = 2    # number of bearings\nCD1 = 3   # node - disc 1 (Python 0-indexed: MATLAB CD1=4)\nCD2 = 9   # node - disc 2 (Python 0-indexed: MATLAB CD2=10)\nCMM1 = 0  # shaft node - bearing 1 (Python 0-indexed: MATLAB CMM1=1)\nCMM2 = 12 # shaft node - bearing 2 (Python 0-indexed: MATLAB CMM2=13)\n\n# Housing nodes (extra nodes beyond shaft nodes)\n# Shaft has NE+1 = 13 nodes (0 to 12)\n# Housing nodes are added as extra nodes\nHOUSING1 = NE + 1  # node 13\nHOUSING2 = NE + 2  # node 14\n\n# =========================================================================\n#   CONSTANTS\n# =========================================================================\nE = 2.0e11    # elasticity modulus [N/m^2]\nRAco = 7800   # steel density [kg/m^3]\nRAl = 2770    # aluminum density [kg/m^3]\n\n# =========================================================================\n#   OPERATIONAL CONDITIONS\n# =========================================================================\nOmega = 0 * 2 * np.pi  # angular velocity [rad/s]\nOmegarpm = Omega * 60 / 2 / np.pi\n\n# =========================================================================\n#   GEOMETRY OF THE ROTATING MACHINE\n# =========================================================================\n\n# (A) DISCS\nRd = 6 / 100                                # external radius of the disc [m]\nRi = (5 / 2) / 1000                         # internal radius of the disc [m]\nespD = 1.1 / 100                            # disc thickness [m]\nMasD = np.pi * Rd**2 * espD * RAl - np.pi * Ri**2 * espD * RAl  # disc mass [kg]\nId = (1/4 * Rd**2 + 1/12 * espD**2) * np.pi * Rd**2 * espD * RAl \\\n   - (1/4 * Ri**2 + 1/12 * espD**2) * np.pi * Ri**2 * espD * RAl  # transversal MOI [kg*m^2]\nIp = 1/2 * Rd * Rd * (np.pi * Rd**2 * espD * RAl) \\\n   - 1/2 * Ri * Ri * (np.pi * Ri**2 * espD * RAl)  # polar MOI [kg*m^2]\n\nprint(f\"Disc mass: {MasD:.4f} kg\")\nprint(f\"Disc Id:   {Id:.6f} kg*m^2\")\nprint(f\"Disc Ip:   {Ip:.6f} kg*m^2\")\n\n# (B) BEARINGS\nMasM = 0.40698   # bearing mass [kg] (housing + ball bearings)\nh = 1 / 1000     # beam thickness [m]\nb = 28.5 / 1000  # beam width [m]\nArea = b * h      # beam cross section area [m^2]\nI_beam = b * h**3 / 12  # beam moment of inertia of area [m^4]\nlr = 7.5 / 100   # beam length [m]\nKty0 = 2 * 12 * E * I_beam / lr**3  # equivalent beam flexural stiffness [N/m]\nKtz0 = 2 * E * Area / lr            # equivalent bar stiffness [N/m]\n\nprint(f\"\\nBearing stiffness Kty0: {Kty0:.2f} N/m\")\nprint(f\"Bearing stiffness Ktz0: {Ktz0:.2f} N/m\")\nprint(f\"Bearing housing mass: {MasM:.4f} kg\")\n\n# (C) SHAFT\nRext = (5 / 2) / 1000  # shaft external radius [m]\nRint = (0 / 2) / 1000  # shaft internal radius [m]\n\n# =========================================================================\n#   CREATE MATERIAL\n# =========================================================================\naluminum = rs.Material(name=\"aluminum\", rho=RAl, E=70e9, G_s=27e9)\nsteel = rs.Material(name=\"steel_DTU\", rho=RAco, E=E, G_s=8.0e10)\n\n# =========================================================================\n#   CREATE SHAFT ELEMENTS\n# =========================================================================\nl = np.zeros(NE)\nl[0]  = 0.140 / 3\nl[1]  = 0.140 / 3\nl[2]  = 0.140 / 3\nl[3]  = 0.205 / 6\nl[4]  = 0.205 / 6\nl[5]  = 0.205 / 6\nl[6]  = 0.205 / 6\nl[7]  = 0.205 / 6\nl[8]  = 0.205 / 6\nl[9]  = 0.090 / 3\nl[10] = 0.090 / 3\nl[11] = 0.090 / 3\n\nshaft_elements = []\nfor i in range(NE):\n    shaft_el = rs.ShaftElement(\n        L=l[i],\n        idl=2 * Rint,\n        odl=2 * Rext,\n        material=steel,\n        n=i,\n        shear_effects=True,\n        rotary_inertia=True,\n        gyroscopic=True,\n    )\n    shaft_elements.append(shaft_el)\n\nprint(f\"\\nNumber of shaft elements: {len(shaft_elements)}\")\nprint(f\"Total shaft length: {np.sum(l):.4f} m\")\nprint(f\"Shaft nodes: 0 to {NE} ({NE+1} nodes)\")\nprint(f\"Housing nodes: {HOUSING1}, {HOUSING2}\")\n\n# =========================================================================\n#   CREATE DISK ELEMENTS\n# =========================================================================\ndisk1 = rs.DiskElement.from_geometry(\n    n=CD1, material=aluminum, width=espD, i_d=2 * Ri, o_d=2 * Rd\n)\ndisk2 = rs.DiskElement.from_geometry(\n    n=CD2, material=aluminum, width=espD, i_d=2 * Ri, o_d=2 * Rd\n)\n\nprint(f\"\\nDisk 1 at node {CD1}\")\nprint(f\"Disk 2 at node {CD2}\")\n\n# =========================================================================\n#   CREATE BEARING ELEMENTS (proper ROSS pattern with n_link)\n# =========================================================================\n# Bearing 1: connects shaft node CMM1 (0) to housing node HOUSING1 (13)\nbearing1 = rs.BearingElement(\n    n=CMM1, \n    n_link=HOUSING1,\n    kxx=Ktz0,  # horizontal stiffness (MATLAB Ktz1)\n    kyy=Kty0,  # vertical stiffness (MATLAB Kty1)\n    cxx=0.0,\n    cyy=0.0,\n    tag=\"Bearing1\"\n)\n\n# Support bearing 1 at housing node HOUSING1 (stiffness to ground)\nsupport1 = rs.BearingElement(\n    n=HOUSING1,\n    kxx=Ktz0,\n    kyy=Kty0,\n    cxx=0.0,\n    cyy=0.0,\n    tag=\"Support1\"\n)\n\n# Bearing 2: connects shaft node CMM2 (12) to housing node HOUSING2 (14)\nbearing2 = rs.BearingElement(\n    n=CMM2,\n    n_link=HOUSING2,\n    kxx=Ktz0,\n    kyy=Kty0,\n    cxx=0.0,\n    cyy=0.0,\n    tag=\"Bearing2\"\n)\n\n# Support bearing 2 at housing node HOUSING2 (stiffness to ground)\nsupport2 = rs.BearingElement(\n    n=HOUSING2,\n    kxx=Ktz0,\n    kyy=Kty0,\n    cxx=0.0,\n    cyy=0.0,\n    tag=\"Support2\"\n)\n\nprint(f\"\\nBearing 1 at shaft node {CMM1} -> housing node {HOUSING1}\")\nprint(f\"Support 1 at housing node {HOUSING1}\")\nprint(f\"Bearing 2 at shaft node {CMM2} -> housing node {HOUSING2}\")\nprint(f\"Support 2 at housing node {HOUSING2}\")\n\n# =========================================================================\n#   CREATE POINT MASSES AT HOUSING NODES\n# =========================================================================\npoint_mass1 = rs.PointMass(n=HOUSING1, m=MasM, tag=\"HousingMass1\")\npoint_mass2 = rs.PointMass(n=HOUSING2, m=MasM, tag=\"HousingMass2\")\n\nprint(f\"\\nPointMass 1 at housing node {HOUSING1}: {MasM:.4f} kg\")\nprint(f\"PointMass 2 at housing node {HOUSING2}: {MasM:.4f} kg\")\n\n# =========================================================================\n#   ASSEMBLE ROTOR\n# =========================================================================\nrotor = rs.Rotor(\n    shaft_elements=shaft_elements,\n    disk_elements=[disk1, disk2],\n    bearing_elements=[bearing1, support1, bearing2, support2],\n    point_mass_elements=[point_mass1, point_mass2],\n)\n\nprint(f\"\\nRotor assembled successfully!\")\nprint(f\"Number of nodes: {len(rotor.nodes)}\")\nprint(f\"Number of DOFs:  {rotor.ndof}\")\n\n# =========================================================================\n#   CONVERT TO 4 DOF and run MODAL ANALYSIS\n# =========================================================================\nfrom ross.utils import convert_6dof_to_4dof\nrotor_4dof = convert_6dof_to_4dof(rotor)\n\nprint(f\"\\nConverted to 4 DOF model\")\nprint(f\"Number of DOFs:  {rotor_4dof.ndof}\")\n\n# Debug: check mass matrix singularity\nM_test = rotor_4dof.M()\nprint(f\"\\nMass matrix shape: {M_test.shape}\")\nprint(f\"Mass matrix min eigenvalue: {np.min(np.abs(np.linalg.eigvalsh(M_test))):.6e}\")\nprint(f\"Mass matrix condition number: {np.linalg.cond(M_test):.6e}\")\n\n# =========================================================================\n#   MODAL ANALYSIS\n# =========================================================================\nprint(\"\\n\" + \"=\" * 60)\nprint(\"RUNNING MODAL ANALYSIS\")\nprint(\"=\" * 60)\n\nmodal = rotor_4dof.run_modal(speed=Omega)\n\nwn_hz = modal.wn / (2 * np.pi)\n\nprint(\"\\nFirst 10 Natural frequencies (Hz):\")\nfor i, freq in enumerate(wn_hz[:10]):\n    print(f\"  Mode {i + 1}: {freq:.2f} Hz\")\n\nprint(\"\\n\" + \"=\" * 60)\nprint(\"COMPARISON WITH EXPERIMENTAL RESULTS\")\nprint(\"=\" * 60)\nprint(\"Experimental results (Hz): 13.0, 14.9, 33.6, 43.0, 46.0\")\nprint(\"ROSS results (Hz):\")\nfor i, freq in enumerate(wn_hz[:10]):\n    print(f\"  Mode {i + 1}: {freq:.2f} Hz\")\n\n# =========================================================================\n#   PRINT TABLE OF RESULTS\n# =========================================================================\nprint(\"\\n\" + \"=\" * 60)\nprint(\"DETAILED MODAL RESULTS\")\nprint(\"=\" * 60)\nprint(modal.format_table())\n\n# =========================================================================\n#   PLOT MODE SHAPES\n# =========================================================================\ntry:\n    for mode_num in range(1, min(7, len(wn_hz) + 1)):\n        try:\n            fig = modal.plot_mode_3d(mode=mode_num)\n            fig.update_layout(title=f\"Mode {mode_num} - f={wn_hz[mode_num - 1]:.2f} Hz\")\n            fig.show()\n        except Exception as e:\n            print(f\"  Mode {mode_num}: Could not plot 3D ({e})\")\n            data = modal.data_mode(mode=mode_num)\n            print(f\"  Mode shape data available at modal.data_mode(mode={mode_num})\")\nexcept Exception as e:\n    print(f\"\\nNote: 3D mode plotting requires plotly display: {e}\")\n    print(\"Use modal.data_mode(mode) to get mode shape data\")\n\n# =========================================================================\n#   RESULTS DATAFRAME\n# =========================================================================\nresults_data = {\n    \"Mode\": range(1, len(wn_hz) + 1),\n    \"wn (rad/s)\": modal.wn,\n    \"wd (rad/s)\": modal.wd,\n    \"Frequency (Hz)\": wn_hz,\n    \"Damping ratio\": modal.damping_ratio,\n    \"Log decrement\": modal.log_dec,\n}\n\ndf_results = pd.DataFrame(results_data)\nprint(\"\\n\" + \"=\" * 60)\nprint(\"RESULTS DATAFRAME\")\nprint(\"=\" * 60)\nprint(df_results.to_string(index=False))\n\n# =========================================================================\n#   VALIDATION\n# =========================================================================\nprint(\"\\n\" + \"=\" * 60)\nprint(\"VALIDATION\")\nprint(\"=\" * 60)\nprint(\"The experimental results from DTU are:\")\nprint(\"  13.0 Hz (horizontal)\")\nprint(\"  14.9 Hz (vertical)\")\nprint(\"  33.6 Hz (horizontal)\")\nprint(\"  43.0 Hz (horizontal)\")\nprint(\"  46.0 Hz (vertical)\")\nprint()\nprint(\"Note: ROSS uses 6 DOFs per node (including axial and torsional),\")\nprint(\"while the MATLAB code uses 4 DOFs (lateral only).\")\nprint(\"Using proper ROSS pattern with n_link for bearing housing modeling.\")";
function copyGabarito(){
navigator.clipboard.writeText(document.getElementById("geral-code").innerText)
.then(()=>alert("Copiado!"));
}
</script>

In [ ]:
"""
MACHINERY DYNAMICS LECTURES  (41514)
MEK - DEPARTMENT OF MECHANICAL ENGINEERING
DTU - TECHNICAL UNIVERSITY OF DENMARK

Copenhagen, March 30th, 2021

Ilmar Ferreira Santos

ROTATING MACHINES -- NATURAL FREQUENCIES AND MODES

EXPERIMENTAL RESULTS
13.0 (horizontal)
14.9 (vertical)
33.6 (horizontal)
43.0 (horizontal)
46.0 (vertical)

Converted from MATLAB to Python using ROSS (Rotordynamic Open-Source Software)
12-element version

Using proper ROSS pattern with n_link:
- Bearing at shaft node with n_link to housing node
- Support bearing at housing node (stiffness to ground)
- PointMass at housing node (housing mass)
"""

import numpy as np
import pandas as pd
import ross as rs
from scipy import linalg as la

# =========================================================================
#   DEFINITION OF THE STRUCTURE OF THE MODEL
# =========================================================================
NE = 12   # number of shaft elements
ND = 2    # number of discs
NM = 2    # number of bearings
CD1 = 3   # node - disc 1 (Python 0-indexed: MATLAB CD1=4)
CD2 = 9   # node - disc 2 (Python 0-indexed: MATLAB CD2=10)
CMM1 = 0  # shaft node - bearing 1 (Python 0-indexed: MATLAB CMM1=1)
CMM2 = 12 # shaft node - bearing 2 (Python 0-indexed: MATLAB CMM2=13)

# Housing nodes (extra nodes beyond shaft nodes)
# Shaft has NE+1 = 13 nodes (0 to 12)
# Housing nodes are added as extra nodes
HOUSING1 = NE + 1  # node 13
HOUSING2 = NE + 2  # node 14

# =========================================================================
#   CONSTANTS
# =========================================================================
E = 2.0e11    # elasticity modulus [N/m^2]
RAco = 7800   # steel density [kg/m^3]
RAl = 2770    # aluminum density [kg/m^3]

# =========================================================================
#   OPERATIONAL CONDITIONS
# =========================================================================
Omega = 0 * 2 * np.pi  # angular velocity [rad/s]
Omegarpm = Omega * 60 / 2 / np.pi

# =========================================================================
#   GEOMETRY OF THE ROTATING MACHINE
# =========================================================================

# (A) DISCS
Rd = 6 / 100                                # external radius of the disc [m]
Ri = (5 / 2) / 1000                         # internal radius of the disc [m]
espD = 1.1 / 100                            # disc thickness [m]
MasD = np.pi * Rd**2 * espD * RAl - np.pi * Ri**2 * espD * RAl  # disc mass [kg]
Id = (1/4 * Rd**2 + 1/12 * espD**2) * np.pi * Rd**2 * espD * RAl \
   - (1/4 * Ri**2 + 1/12 * espD**2) * np.pi * Ri**2 * espD * RAl  # transversal MOI [kg*m^2]
Ip = 1/2 * Rd * Rd * (np.pi * Rd**2 * espD * RAl) \
   - 1/2 * Ri * Ri * (np.pi * Ri**2 * espD * RAl)  # polar MOI [kg*m^2]

print(f"Disc mass: {MasD:.4f} kg")
print(f"Disc Id:   {Id:.6f} kg*m^2")
print(f"Disc Ip:   {Ip:.6f} kg*m^2")

# (B) BEARINGS
MasM = 0.40698   # bearing mass [kg] (housing + ball bearings)
h = 1 / 1000     # beam thickness [m]
b = 28.5 / 1000  # beam width [m]
Area = b * h      # beam cross section area [m^2]
I_beam = b * h**3 / 12  # beam moment of inertia of area [m^4]
lr = 7.5 / 100   # beam length [m]
Kty0 = 2 * 12 * E * I_beam / lr**3  # equivalent beam flexural stiffness [N/m]
Ktz0 = 2 * E * Area / lr            # equivalent bar stiffness [N/m]

print(f"\nBearing stiffness Kty0: {Kty0:.2f} N/m")
print(f"Bearing stiffness Ktz0: {Ktz0:.2f} N/m")
print(f"Bearing housing mass: {MasM:.4f} kg")

# (C) SHAFT
Rext = (5 / 2) / 1000  # shaft external radius [m]
Rint = (0 / 2) / 1000  # shaft internal radius [m]

# =========================================================================
#   CREATE MATERIAL
# =========================================================================
aluminum = rs.Material(name="aluminum", rho=RAl, E=70e9, G_s=27e9)
steel = rs.Material(name="steel_DTU", rho=RAco, E=E, G_s=8.0e10)

# =========================================================================
#   CREATE SHAFT ELEMENTS
# =========================================================================
l = np.zeros(NE)
l[0]  = 0.140 / 3
l[1]  = 0.140 / 3
l[2]  = 0.140 / 3
l[3]  = 0.205 / 6
l[4]  = 0.205 / 6
l[5]  = 0.205 / 6
l[6]  = 0.205 / 6
l[7]  = 0.205 / 6
l[8]  = 0.205 / 6
l[9]  = 0.090 / 3
l[10] = 0.090 / 3
l[11] = 0.090 / 3

shaft_elements = []
for i in range(NE):
    shaft_el = rs.ShaftElement(
        L=l[i],
        idl=2 * Rint,
        odl=2 * Rext,
        material=steel,
        n=i,
        shear_effects=True,
        rotary_inertia=True,
        gyroscopic=True,
    )
    shaft_elements.append(shaft_el)

print(f"\nNumber of shaft elements: {len(shaft_elements)}")
print(f"Total shaft length: {np.sum(l):.4f} m")
print(f"Shaft nodes: 0 to {NE} ({NE+1} nodes)")
print(f"Housing nodes: {HOUSING1}, {HOUSING2}")

# =========================================================================
#   CREATE DISK ELEMENTS
# =========================================================================
disk1 = rs.DiskElement.from_geometry(
    n=CD1, material=aluminum, width=espD, i_d=2 * Ri, o_d=2 * Rd
)
disk2 = rs.DiskElement.from_geometry(
    n=CD2, material=aluminum, width=espD, i_d=2 * Ri, o_d=2 * Rd
)

print(f"\nDisk 1 at node {CD1}")
print(f"Disk 2 at node {CD2}")

# =========================================================================
#   CREATE BEARING ELEMENTS (proper ROSS pattern with n_link)
# =========================================================================
# Bearing 1: connects shaft node CMM1 (0) to housing node HOUSING1 (13)
bearing1 = rs.BearingElement(
    n=CMM1, 
    n_link=HOUSING1,
    kxx=Ktz0,  # horizontal stiffness (MATLAB Ktz1)
    kyy=Kty0,  # vertical stiffness (MATLAB Kty1)
    cxx=0.0,
    cyy=0.0,
    tag="Bearing1"
)

# Support bearing 1 at housing node HOUSING1 (stiffness to ground)
support1 = rs.BearingElement(
    n=HOUSING1,
    kxx=Ktz0,
    kyy=Kty0,
    cxx=0.0,
    cyy=0.0,
    tag="Support1"
)

# Bearing 2: connects shaft node CMM2 (12) to housing node HOUSING2 (14)
bearing2 = rs.BearingElement(
    n=CMM2,
    n_link=HOUSING2,
    kxx=Ktz0,
    kyy=Kty0,
    cxx=0.0,
    cyy=0.0,
    tag="Bearing2"
)

# Support bearing 2 at housing node HOUSING2 (stiffness to ground)
support2 = rs.BearingElement(
    n=HOUSING2,
    kxx=Ktz0,
    kyy=Kty0,
    cxx=0.0,
    cyy=0.0,
    tag="Support2"
)

print(f"\nBearing 1 at shaft node {CMM1} -> housing node {HOUSING1}")
print(f"Support 1 at housing node {HOUSING1}")
print(f"Bearing 2 at shaft node {CMM2} -> housing node {HOUSING2}")
print(f"Support 2 at housing node {HOUSING2}")

# =========================================================================
#   CREATE POINT MASSES AT HOUSING NODES
# =========================================================================
point_mass1 = rs.PointMass(n=HOUSING1, m=MasM, tag="HousingMass1")
point_mass2 = rs.PointMass(n=HOUSING2, m=MasM, tag="HousingMass2")

print(f"\nPointMass 1 at housing node {HOUSING1}: {MasM:.4f} kg")
print(f"PointMass 2 at housing node {HOUSING2}: {MasM:.4f} kg")

# =========================================================================
#   ASSEMBLE ROTOR
# =========================================================================
rotor = rs.Rotor(
    shaft_elements=shaft_elements,
    disk_elements=[disk1, disk2],
    bearing_elements=[bearing1, support1, bearing2, support2],
    point_mass_elements=[point_mass1, point_mass2],
)

print(f"\nRotor assembled successfully!")
print(f"Number of nodes: {len(rotor.nodes)}")
print(f"Number of DOFs:  {rotor.ndof}")

# =========================================================================
#   CONVERT TO 4 DOF and run MODAL ANALYSIS
# =========================================================================
from ross.utils import convert_6dof_to_4dof
rotor_4dof = convert_6dof_to_4dof(rotor)

print(f"\nConverted to 4 DOF model")
print(f"Number of DOFs:  {rotor_4dof.ndof}")

# Debug: check mass matrix singularity
M_test = rotor_4dof.M()
print(f"\nMass matrix shape: {M_test.shape}")
print(f"Mass matrix min eigenvalue: {np.min(np.abs(np.linalg.eigvalsh(M_test))):.6e}")
print(f"Mass matrix condition number: {np.linalg.cond(M_test):.6e}")

# =========================================================================
#   MODAL ANALYSIS
# =========================================================================
print("\n" + "=" * 60)
print("RUNNING MODAL ANALYSIS")
print("=" * 60)

modal = rotor_4dof.run_modal(speed=Omega)

wn_hz = modal.wn / (2 * np.pi)

print("\nFirst 10 Natural frequencies (Hz):")
for i, freq in enumerate(wn_hz[:10]):
    print(f"  Mode {i + 1}: {freq:.2f} Hz")

print("\n" + "=" * 60)
print("COMPARISON WITH EXPERIMENTAL RESULTS")
print("=" * 60)
print("Experimental results (Hz): 13.0, 14.9, 33.6, 43.0, 46.0")
print("ROSS results (Hz):")
for i, freq in enumerate(wn_hz[:10]):
    print(f"  Mode {i + 1}: {freq:.2f} Hz")

# =========================================================================
#   PRINT TABLE OF RESULTS
# =========================================================================
print("\n" + "=" * 60)
print("DETAILED MODAL RESULTS")
print("=" * 60)
print(modal.format_table())

# =========================================================================
#   PLOT MODE SHAPES
# =========================================================================
try:
    for mode_num in range(1, min(7, len(wn_hz) + 1)):
        try:
            fig = modal.plot_mode_3d(mode=mode_num)
            fig.update_layout(title=f"Mode {mode_num} - f={wn_hz[mode_num - 1]:.2f} Hz")
            fig.show()
        except Exception as e:
            print(f"  Mode {mode_num}: Could not plot 3D ({e})")
            data = modal.data_mode(mode=mode_num)
            print(f"  Mode shape data available at modal.data_mode(mode={mode_num})")
except Exception as e:
    print(f"\nNote: 3D mode plotting requires plotly display: {e}")
    print("Use modal.data_mode(mode) to get mode shape data")

# =========================================================================
#   RESULTS DATAFRAME
# =========================================================================
results_data = {
    "Mode": range(1, len(wn_hz) + 1),
    "wn (rad/s)": modal.wn,
    "wd (rad/s)": modal.wd,
    "Frequency (Hz)": wn_hz,
    "Damping ratio": modal.damping_ratio,
    "Log decrement": modal.log_dec,
}

df_results = pd.DataFrame(results_data)
print("\n" + "=" * 60)
print("RESULTS DATAFRAME")
print("=" * 60)
print(df_results.to_string(index=False))

# =========================================================================
#   VALIDATION
# =========================================================================
print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)
print("The experimental results from DTU are:")
print("  13.0 Hz (horizontal)")
print("  14.9 Hz (vertical)")
print("  33.6 Hz (horizontal)")
print("  43.0 Hz (horizontal)")
print("  46.0 Hz (vertical)")
print()
print("Note: ROSS uses 6 DOFs per node (including axial and torsional),")
print("while the MATLAB code uses 4 DOFs (lateral only).")
print("Using proper ROSS pattern with n_link for bearing housing modeling.")